# LogLens AI: Security Log Analyzer

**LogLens AI** reads exported security / event logs, sends them in small batches to the class **Qwen Coder** model
(hosted on the UTSA Compute Hub GPUStack), and classifies every entry as:

| Label | Meaning |
|---|---|
| **BENIGN** | Normal or harmless activity |
| **SUSPICIOUS** | May need further investigation |
| **MALICIOUS** | Looks harmful or associated with an attack |

Each classification comes with a short explanation, and the tool writes a Markdown **security assessment report**.

> **Security design:** the API key is read at runtime from the `GPUSTACK_API_KEY` environment variable or a hidden
> prompt and is never written to the notebook. Raw logs are excluded from version control (`.gitignore`); only the
> redacted assessment report is published.

## Step 1: Setup

In [1]:
%pip install --quiet openai

Note: you may need to restart the kernel to use updated packages.


In [2]:
import os, re, csv, json, time, getpass
from collections import Counter
from datetime import datetime
from openai import OpenAI

# GPUStack exposes an OpenAI-compatible API.
GPUSTACK_URL = "http://gpustack.cds.utsa.edu"

# The key comes from the environment or a hidden prompt, so it is never stored in the notebook.
API_KEY = os.environ.get("GPUSTACK_API_KEY") or getpass.getpass("GPUStack API key: ")

GPUStack API key:  ········


## Step 2: Connect to Qwen Coder

In [3]:
def connect(base_url_root, api_key):
    # Newer GPUStack versions serve /v1, older ones /v1-openai; try both.
    last_err = None
    for suffix in ("/v1", "/v1-openai"):
        client = OpenAI(base_url=base_url_root + suffix, api_key=api_key, timeout=60)
        try:
            models = [m.id for m in client.models.list().data]
            return client, models
        except Exception as e:
            last_err = e
    raise ConnectionError(f"Could not reach GPUStack (on VPN / campus network?): {last_err}")

client, available_models = connect(GPUSTACK_URL, API_KEY)
print("Available models:", available_models)

# Prefer a Qwen Coder model, otherwise any Qwen model, otherwise the first one listed.
MODEL = next((m for m in available_models if "qwen" in m.lower() and "coder" in m.lower()),
             next((m for m in available_models if "qwen" in m.lower()), available_models[0]))

reply = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": "Reply with one short sentence confirming you are online."}],
    max_tokens=40,
)
print(f"Connected to model: {MODEL}")
print("Model says:", reply.choices[0].message.content.strip())

Available models: ['qwen3-coder-next', 'qwen3.5-122b-a10b', 'qwen3-vl-embedding-8b', 'qwen3-vl-32b-instruct']
Connected to model: qwen3-coder-next
Model says: Yes, I’m online and ready to help!


## Step 3: Tool design with Qwen Coder

Qwen Coder is prompted for a design of the analyzer. The design points used in Step 4 are batching entries,
strict JSON output, label validation, a safe fallback for unparseable output, and example event types in the prompt.

In [4]:
def ask_qwen(prompt, max_tokens=900):
    r = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": prompt}],
        temperature=0.2,
        max_tokens=max_tokens,
    )
    return r.choices[0].message.content

design_prompt = '''I am building a Python tool called "LogLens AI" that uses an LLM to classify Windows/macOS
security log entries as BENIGN, SUSPICIOUS, or MALICIOUS with a one-sentence explanation each.
Suggest a design: how to load CSV log exports, how to batch entries, what prompt and output format to use so the
results are machine-readable, how to handle bad model output, and what a final security assessment report should
contain. Keep it concise, as a numbered list, with a short code sketch for the prompt.'''

print(ask_qwen(design_prompt))

1. **CSV Loading**: Use `pandas.read_csv()` with `dtype=str` to preserve raw log fields; handle missing/empty fields by substituting `"N/A"`.

2. **Batching**: Group entries into batches of 20–50 logs (tunable) to stay within token limits and improve throughput; sort by timestamp for chronological context.

3. **Prompt Design** (JSON-structured, few-shot examples included):
   ```python
   SYSTEM_PROMPT = """You are a security analyst. Classify each log entry as BENIGN, SUSPICIOUS, or MALICIOUS.
   Output ONLY valid JSON: {"results": [{"id": "<log_id>", "classification": "<CLASS>", "explanation": "<1-sentence>"}]}."""
   
   USER_PROMPT = """Analyze these Windows/macOS security logs. Use only the provided fields.

   [LOGS]
   {log_batch_json}
   [/LOGS]

   Return JSON with exactly one result per log, preserving original IDs."""
   ```

4. **Output Format**: Enforce strict JSON schema; parse with `json.loads()`. Validate each result has `id`, `classification` (enum), and `explanation`

## Step 4: LogLens AI tool code

How it works:
1. **`load_log`** reads a CSV export (Windows Event Viewer / PowerShell `Export-Csv`) or a plain-text log (macOS `log show`), one entry per row/line.
2. **`classify_batch`** sends ~10 entries at a time with a system prompt that demands a JSON array of
   `{id, classification, explanation}` objects.
3. **`parse_response`** extracts the JSON (repairing unescaped Windows-path backslashes) and checks every label is
   one of the three allowed values. Entries a batch call missed are retried one at a time; anything still
   unparseable is marked **SUSPICIOUS** ("needs manual review") rather than silently dropped.
   **`confirm_malicious`** re-checks each MALICIOUS entry on its own; a label that is not confirmed drops to
   SUSPICIOUS so a single inconsistent model response cannot set the overall risk rating.
4. **`build_report`** writes the Markdown assessment. **`redact`** strips SIDs, account names, user-profile paths,
   e-mail addresses and IP addresses so the report is safe to publish.

**Prompt calibration.** Routine Windows events (e.g. DCOM 10016 permission warnings, which Microsoft documents as
safe to ignore) are frequent and easily over-classified. The system prompt lists common benign event types and
requires concrete evidence before an entry is labeled SUSPICIOUS or MALICIOUS.

In [5]:
LABELS = ("BENIGN", "SUSPICIOUS", "MALICIOUS")
BATCH_SIZE = 10
MAX_ENTRY_CHARS = 500

SYSTEM_PROMPT = '''You are a senior SOC (security operations center) analyst.
You will receive numbered security log entries from a Windows or macOS computer.
Classify EACH entry as exactly one of: BENIGN, SUSPICIOUS, MALICIOUS.
- BENIGN: routine, expected system activity.
- SUSPICIOUS: unusual or security-relevant activity that a human should look at.
- MALICIOUS: strong indicators of an attack (e.g. malware detection, brute force, log clearing, unknown persistence).
Calibrate carefully; most event log entries are routine noise:
- A Warning or Error level alone is NOT evidence of an attack. Judge the content of the entry.
- Well-known Windows noise is BENIGN unless the entry itself shows attack indicators: DCOM 10016/10010
  permission or timeout warnings, failed Store/Windows Update installs, time-sync DNS failures, BitLocker
  boot/restart key messages, service start-type changes, Defender signature and configuration updates.
- Repetition of the same routine warning does not make it more suspicious.
- SUSPICIOUS needs something specific worth a human look (e.g. a new service from an unusual path,
  a security feature being turned off, an unexpected crash of a security product).
- MALICIOUS needs clear evidence, not a hypothetical "could indicate".
Respond with ONLY a JSON array, no prose, no markdown fences, in this form:
[{"id": <entry id>, "classification": "BENIGN|SUSPICIOUS|MALICIOUS", "explanation": "<one sentence>"}]'''


def load_log(path):
    # CSV exports become dicts; anything else is one entry per non-empty line.
    entries = []
    if path.lower().endswith(".csv"):
        with open(path, newline="", encoding="utf-8-sig", errors="replace") as f:
            for row in csv.DictReader(f):
                text = " | ".join(f"{k}: {v}" for k, v in row.items() if v)
                entries.append({"id": len(entries) + 1, "raw": row, "text": text})
    else:
        with open(path, encoding="utf-8", errors="replace") as f:
            for line in f:
                if line.strip():
                    entries.append({"id": len(entries) + 1, "raw": {"Message": line.strip()}, "text": line.strip()})
    for e in entries:
        e["text"] = e["text"][:MAX_ENTRY_CHARS]
    return entries


def parse_response(content, expected_ids):
    # Pull the first JSON array out of the reply (models sometimes wrap it in ``` fences or prose).
    match = re.search(r"\[.*\]", content or "", re.S)
    if not match:
        raise ValueError("no JSON array in response")
    raw = match.group(0)
    try:
        items = json.loads(raw)
    except json.JSONDecodeError:
        # Models often echo Windows paths (C:\Windows\...) with unescaped backslashes; escape them and retry.
        items = json.loads(re.sub(r'\\(?![\\"/bfnrtu])', r"\\\\", raw))
    results = {}
    for item in items:
        label = str(item.get("classification", "")).upper().strip()
        eid = int(item.get("id", -1))
        if eid in expected_ids and label in LABELS:
            results[eid] = {"classification": label, "explanation": str(item.get("explanation", "")).strip()}
    if not results:
        raise ValueError("JSON contained no valid classifications")
    return results


def request_labels(batch):
    ids = {e["id"] for e in batch}
    r = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "system", "content": SYSTEM_PROMPT},
                  {"role": "user", "content": "\n".join(f"[{e['id']}] {e['text']}" for e in batch)}],
        temperature=0,
        max_tokens=200 * len(batch),
    )
    return parse_response(r.choices[0].message.content, ids)


def classify_batch(batch):
    ids = {e["id"] for e in batch}
    results = {}
    try:
        results.update(request_labels(batch))
    except Exception as err:
        print(f"  batch {min(ids)}-{max(ids)} failed ({err}); retrying entries one at a time")
    # Anything the batch call missed gets its own request.
    for e in batch:
        if e["id"] in results:
            continue
        try:
            results.update(request_labels([e]))
        except Exception as err:
            print(f"  entry {e['id']} failed: {err}")
            time.sleep(1)
    for eid in ids - results.keys():
        results[eid] = {"classification": "SUSPICIOUS",
                        "explanation": "Model output could not be parsed for this entry; needs manual review."}
    return results


def analyze_log(path):
    entries = load_log(path)
    print(f"Loaded {len(entries)} entries from {path}")
    for i in range(0, len(entries), BATCH_SIZE):
        batch = entries[i:i + BATCH_SIZE]
        print(f"Classifying entries {batch[0]['id']}-{batch[-1]['id']} ...")
        verdicts = classify_batch(batch)
        for e in batch:
            e.update(verdicts[e["id"]])
    confirm_malicious(entries)
    return entries


def confirm_malicious(entries):
    # A MALICIOUS label must be confirmed by a second, single-entry review; otherwise it drops to SUSPICIOUS.
    for e in entries:
        if e["classification"] != "MALICIOUS":
            continue
        try:
            second = request_labels([e]).get(e["id"])
        except Exception as err:
            second = None
            print(f"  second review of entry {e['id']} failed: {err}")
        if not second or second["classification"] != "MALICIOUS":
            print(f"  entry {e['id']}: MALICIOUS not confirmed on second review; downgraded to SUSPICIOUS")
            e["classification"] = "SUSPICIOUS"
            # The explanation records both reviews so the SUSPICIOUS label and its reason stay consistent.
            if second:
                e["explanation"] = (f"Rated MALICIOUS in the batch review but {second['classification']} on a second "
                                    f"review ({second['explanation'].rstrip('.')}); conflicting results, flagged for "
                                    "analyst review.")
            else:
                e["explanation"] = ("Rated MALICIOUS in the batch review, but the second review could not be "
                                    f"completed; flagged for analyst review. First review: {e['explanation']}")


REDACTIONS = [
    (re.compile(r"S-1-5-21-[\d-]+"), "S-1-5-21-<redacted>"),
    (re.compile(r"(?i)([A-Z]:\\Users\\)[^\\\s|]+"), r"\1<user>"),
    (re.compile(r"/Users/[^/\s|]+"), "/Users/<user>"),
    # COMPUTER\user account names (but not path segments like C:\Windows\System32 or NT AUTHORITY\SYSTEM)
    (re.compile(r"(?<![\\:\w])(?<!NT )[\w.-]+\\[\w-]+(?:\.[\w-]+)*(?![\\\w])"), "<host>\\\\<user>"),
    (re.compile(r"[\w.+-]+@[\w-]+\.[\w.]+"), "<email>"),
    (re.compile(r"\b(?:\d{1,3}\.){3}\d{1,3}\b"), "<ip>"),
]

def redact(text):
    for pattern, repl in REDACTIONS:
        text = pattern.sub(repl, text)
    return text


def short_event(e):
    raw = e["raw"]
    when = raw.get("TimeCreated", "")
    source = raw.get("Provider") or raw.get("LogName") or "log"
    eid = raw.get("EventId", "")
    return redact(f"{when} | {source}" + (f" (Event {eid})" if eid else ""))


def build_report(entries, tool_name="LogLens AI", source_desc="Windows event log export"):
    counts = Counter(e["classification"] for e in entries)
    total = len(entries)
    if counts["MALICIOUS"]:
        risk = "HIGH"
    elif counts["SUSPICIOUS"] > max(3, total * 0.15):
        risk = "MEDIUM"
    elif counts["SUSPICIOUS"]:
        risk = "LOW-MEDIUM"
    else:
        risk = "LOW"

    findings_prompt = ("Act as a SOC analyst. Based on these classified log findings, write 3-5 short, practical "
                       "recommendations for the computer owner as a markdown bullet list. No preamble.\n\n" +
                       "\n".join(f"- {e['classification']}: {short_event(e)}: {e['explanation']}"
                                 for e in entries if e["classification"] != "BENIGN")[:6000])
    try:
        recommendations = ask_qwen(findings_prompt, max_tokens=500).strip() if counts["SUSPICIOUS"] + counts["MALICIOUS"] else \
            "- No action required; continue routine monitoring and keep the system updated."
    except Exception as err:
        recommendations = f"- (Recommendations unavailable: {err})"

    lines = [
        f"# {tool_name}: Security Assessment Report",
        "",
        f"- **Generated:** {datetime.now():%Y-%m-%d %H:%M}",
        f"- **Analysis model:** {MODEL} (UTSA Compute Hub GPUStack)",
        f"- **Log source:** {source_desc}",
        f"- **Entries analyzed:** {total}",
        f"- **Overall risk rating:** **{risk}**",
        "",
        "## Summary",
        "",
        "| Classification | Count | Share |",
        "|---|---|---|",
    ]
    for label in LABELS:
        lines.append(f"| {label} | {counts[label]} | {counts[label] / total:.0%} |" if total else f"| {label} | 0 | - |")

    for label in ("MALICIOUS", "SUSPICIOUS"):
        flagged = [e for e in entries if e["classification"] == label]
        lines += ["", f"## {label.title()} findings ({len(flagged)})", ""]
        if not flagged:
            lines.append("None.")
        for e in flagged:
            lines.append(f"- **#{e['id']}** {short_event(e)}: {redact(e['explanation'])}")

    benign = Counter(short_event(e).split(" | ", 1)[-1] for e in entries if e["classification"] == "BENIGN")
    lines += ["", "## Benign activity (grouped)", "", "| Source | Entries |", "|---|---|"]
    lines += [f"| {src} | {n} |" for src, n in benign.most_common()]

    lines += ["", "## Recommendations", "", redact(recommendations), "",
              "## Method", "",
              f"Entries were sent to the model in batches of {BATCH_SIZE} with a SOC-analyst system prompt requiring strict "
              "JSON output. Labels were validated against the three allowed values; entries the model could not classify "
              "were conservatively marked SUSPICIOUS. Each MALICIOUS label was confirmed by a second single-entry review. Identifiers (SIDs, account names, user paths, e-mail and IP addresses) are redacted. "
              "AI classifications are a triage aid and should be verified by a human analyst.", ""]
    return "\n".join(lines)

## Step 5: Load the security log

`LOG_PATH` names an exported log file in the notebook's directory. Supported export formats:

- **Windows (PowerShell, CSV):**
  `Get-WinEvent -LogName System -MaxEvents 60 | Select TimeCreated,LogName,Id,LevelDisplayName,ProviderName,Message | Export-Csv windows_security_log.csv -NoTypeInformation`
  (reading the `Security` log requires an elevated PowerShell). Event Viewer → *Save All Events As… → CSV* produces the same format.
- **macOS (plain text):** `log show --last 1h --predicate 'subsystem == "com.apple.securityd"' > mac_security_log.txt`

In [6]:
LOG_PATH = "windows_security_log.csv"

assert os.path.exists(LOG_PATH), f"{LOG_PATH} not found in the notebook directory"
preview = load_log(LOG_PATH)
print(f"{LOG_PATH}: {len(preview)} entries, {os.path.getsize(LOG_PATH):,} bytes")
for e in preview[:3]:
    print(f"[{e['id']}]", redact(e["text"])[:200])

windows_security_log.csv: 60 entries, 20,111 bytes
[1] TimeCreated: 2026-11-02T11:14:36 | LogName: System | EventId: 10016 | Level: Warning | Provider: Microsoft-Windows-DistributedCOM | Message: The application-specific permission settings do not grant L
[2] TimeCreated: 2026-11-02T11:15:07 | LogName: System | EventId: 10016 | Level: Warning | Provider: Microsoft-Windows-DistributedCOM | Message: The application-specific permission settings do not grant L
[3] TimeCreated: 2026-11-02T11:15:57 | LogName: System | EventId: 10016 | Level: Warning | Provider: Microsoft-Windows-DistributedCOM | Message: The application-specific permission settings do not grant L


## Step 6: Analyze the log

In [7]:
results = analyze_log(LOG_PATH)

counts = Counter(e["classification"] for e in results)
print("\nResults:", dict(counts))
for e in results:
    if e["classification"] != "BENIGN":
        print(f"#{e['id']:>3} {e['classification']:<10} {short_event(e)}\n      {redact(e['explanation'])}")

Loaded 60 entries from windows_security_log.csv
Classifying entries 1-10 ...
Classifying entries 11-20 ...
Classifying entries 21-30 ...
Classifying entries 31-40 ...
Classifying entries 41-50 ...
Classifying entries 51-60 ...
  entry 10: MALICIOUS not confirmed on second review; downgraded to SUSPICIOUS

Results: {'BENIGN': 43, 'SUSPICIOUS': 17}
#  8 SUSPICIOUS 2026-11-02T11:59:14 | Microsoft-Windows-DistributedCOM (Event 10010)
      Event ID 10010 indicating DCOM server registration timeout for CLSID {CE6AF8E5-3A75-4AF5-BD59-C42E7228B4F4} (often associated with Windows Defender ATP/EDR components) may indicate interference with security services and warrants investigation.
#  9 SUSPICIOUS 2026-11-02T11:59:15 | Microsoft-Windows-DistributedCOM (Event 10010)
      Event ID 10010 for CLSID {9BA05972-F6A8-11CF-A442-00A0C90A8F39} (Windows Script Host) failing to register may indicate script-based malware or tampering with scripting infrastructure, requiring human review.
# 10 SUSPICIOUS 

## Step 7: Generate the security assessment report

In [8]:
report = build_report(results, source_desc="Windows System + Windows Defender event logs (CSV export)")
REPORT_PATH = "security_assessment_report.md"
with open(REPORT_PATH, "w", encoding="utf-8") as f:
    f.write(report)
print(f"Saved {REPORT_PATH}\n")

from IPython.display import Markdown, display
display(Markdown(report))

Saved security_assessment_report.md



# LogLens AI: Security Assessment Report

- **Generated:** 2026-11-10 21:57
- **Analysis model:** qwen3-coder-next (UTSA Compute Hub GPUStack)
- **Log source:** Windows System + Windows Defender event logs (CSV export)
- **Entries analyzed:** 60
- **Overall risk rating:** **MEDIUM**

## Summary

| Classification | Count | Share |
|---|---|---|
| BENIGN | 43 | 72% |
| SUSPICIOUS | 17 | 28% |
| MALICIOUS | 0 | 0% |

## Malicious findings (0)

None.

## Suspicious findings (17)

- **#8** 2026-11-02T11:59:14 | Microsoft-Windows-DistributedCOM (Event 10010): Event ID 10010 indicating DCOM server registration timeout for CLSID {CE6AF8E5-3A75-4AF5-BD59-C42E7228B4F4} (often associated with Windows Defender ATP/EDR components) may indicate interference with security services and warrants investigation.
- **#9** 2026-11-02T11:59:15 | Microsoft-Windows-DistributedCOM (Event 10010): Event ID 10010 for CLSID {9BA05972-F6A8-11CF-A442-00A0C90A8F39} (Windows Script Host) failing to register may indicate script-based malware or tampering with scripting infrastructure, requiring human review.
- **#10** 2026-11-03T20:24:42 | Microsoft-Windows-BitLocker-Driver (Event 24641): Rated MALICIOUS in the batch review but BENIGN on a second review (BitLocker driver error 24641 during restart is a known benign issue often caused by timing or driver initialization delays and does not indicate an attack); conflicting results, flagged for analyst review.
- **#16** 2026-11-03T20:26:02 | Microsoft-Windows-DistributedCOM (Event 10016): DCOM 10016 warning for a user-specific SID (<host>\<user>) with an unknown CLSID may indicate misconfigured or potentially malicious COM activation attempts requiring human review.
- **#17** 2026-11-03T20:27:05 | Microsoft-Windows-DistributedCOM (Event 10016): Repeated DCOM 10016 warnings for the same CLSID/APPID to a user-specific SID suggest possible unauthorized COM server activation or misconfiguration warranting investigation.
- **#18** 2026-11-03T20:27:06 | Microsoft-Windows-DistributedCOM (Event 10016): Third occurrence of DCOM 10016 for the same CLSID/APPID to the same user indicates persistent attempt, increasing likelihood of misconfiguration or suspicious activity.
- **#19** 2026-11-03T20:35:54 | Microsoft-Windows-DistributedCOM (Event 10016): Repetition of DCOM 10016 for the same CLSID/APPID to the same user after ~10 minutes suggests ongoing activity that may indicate persistence or misconfigured application.
- **#25** 2026-11-04T04:36:57 | Microsoft-Windows-Windows Defender (Event 5007): Defender configuration change (WdConfigHash modification) is flagged as potentially suspicious because it could indicate tampering, though it may also be legitimate (e.g., policy update or update).
- **#34** 2026-11-10T20:45:07 | Microsoft-Windows-BitLocker-Driver (Event 24641): BitLocker volume master key retrieval failure during restart may indicate hardware issues, tampering, or misconfiguration requiring investigation.
- **#35** 2026-11-10T20:45:07 | Microsoft-Windows-BitLocker-Driver (Event 24641): Duplicate of event 34; repeated BitLocker key retrieval failure increases concern for potential security or configuration issue.
- **#39** 2026-11-10T20:45:21 | Microsoft-Windows-Windows Defender (Event 5007): Unexpected change to Windows Defender UX configuration (ToastOrSsoTrigger) may indicate tampering and warrants human review.
- **#40** 2026-11-10T20:46:28 | Service Control Manager (Event 7031): Unexpected termination of the Universal Device Client Service (UDC) may indicate conflict, corruption, or malicious interference requiring investigation.
- **#41** 2026-11-10T20:47:11 | Microsoft-Windows-Windows Defender (Event 5007): Multiple Windows Defender configuration changes occurred simultaneously, including disabling branding and enabling sensor platform parsers, which could indicate tampering and warrants human review.
- **#42** 2026-11-10T20:47:11 | Microsoft-Windows-Windows Defender (Event 5007): Disabling Defender branding (MpFC_EnableDefenderBranding = 0x0) is unusual and may indicate attempts to hide Defender presence or interfere with its operation.
- **#43** 2026-11-10T20:47:11 | Microsoft-Windows-Windows Defender (Event 5007): ETag value change in Defender configuration lacks context and may indicate configuration manipulation, especially when grouped with other suspicious Defender changes.
- **#44** 2026-11-10T20:47:12 | Microsoft-Windows-Windows Defender (Event 5007): Disabling control ID 386 (likely related to exploit protection or attack surface reduction) in Windows Defender is a high-risk change that reduces system security.
- **#45** 2026-11-10T20:47:12 | Microsoft-Windows-Windows Defender (Event 5007): Enabling control ID 285 (possibly related to tamper protection or real-time monitoring) without context is suspicious, especially when combined with other Defender config changes.

## Benign activity (grouped)

| Source | Entries |
|---|---|
| Service Control Manager (Event 7040) | 10 |
| Microsoft-Windows-DistributedCOM (Event 10016) | 9 |
| Service Control Manager (Event 7045) | 5 |
| Microsoft-Windows-Windows Defender (Event 1151) | 3 |
| Microsoft-Windows-NDIS (Event 10317) | 2 |
| Microsoft-Windows-Time-Service (Event 134) | 2 |
| Microsoft-Windows-WindowsUpdateClient (Event 20) | 2 |
| Microsoft-Windows-Windows Defender (Event 2000) | 2 |
| Microsoft-Windows-Windows Defender (Event 1150) | 2 |
| Win32k (Event 700) | 2 |
| Win32k (Event 701) | 2 |
| Microsoft-Windows-BitLocker-Driver (Event 24641) | 1 |
| Microsoft-Windows-Windows Defender (Event 1013) | 1 |

## Recommendations

- **Immediately investigate and restore Windows Defender configuration**: Multiple Defender config changes (e.g., branding disabled, exploit protection disabled, sensor parsers enabled) strongly suggest tampering—run `Get-MpConfiguration -Detailed` and compare against baseline; restore default policies via Group Policy or Microsoft Defender Security Center.
- **Audit COM/DCOM activity for the user `<host>\<user>`**: Repeated DCOM 10016 events for the same CLSID/APPID indicate possible persistence or misconfiguration—review Process Monitor logs, check registry keys under `<host>\<user>` and `AppID`, and verify legitimacy of associated services.
- **Verify BitLocker integrity and hardware health**: Repeated Event 24641 failures during restart may indicate tampering or hardware issues—run `manage-bde -status`, check BitLocker recovery key, and inspect disk/event logs for unauthorized changes or firmware anomalies.
- **Disable and re-enable Windows Script Host (WSH) cautiously**: DCOM registration failure for WSH (CLSID `{9BA05972-F6A8-11CF-A442-00A0C90A8F39}`) may signal script-based malware—temporarily disable WSH via Group Policy (`Prevent execution of script files`), scan with `sigcheck -v`, and re-enable only after confirming system cleanliness.
- **Review and harden service configurations**: Unexpected termination of the Universal Device Client Service (UDC) may indicate conflict or compromise—check `services.msc` for unauthorized changes, verify service integrity via `sc query udc`, and ensure only trusted services are configured to auto-start.

## Method

Entries were sent to the model in batches of 10 with a SOC-analyst system prompt requiring strict JSON output. Labels were validated against the three allowed values; entries the model could not classify were conservatively marked SUSPICIOUS. Each MALICIOUS label was confirmed by a second single-entry review. Identifiers (SIDs, account names, user paths, e-mail and IP addresses) are redacted. AI classifications are a triage aid and should be verified by a human analyst.
